In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from itables import show

df = pd.read_parquet("../data/portfolio_priced.parquet")
df.shape

(668897, 19)

In [2]:
df.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimAmount,age_group,bm_group,freq_pred,sev_pred,prime_pure,prime_pure_v2
66,139.0,1,0.75,F,7,1,61,50,B12,'Regular',27000,R11,303.00,"(55, 65]","(49, 60]",0.053737,1756.351908,122.479400,127.114796
93,190.0,1,0.14,B,12,5,50,60,B12,'Diesel',56,R25,1981.84,"(45, 55]","(49, 60]",0.063583,1806.830859,142.982836,143.917067
199,414.0,1,0.14,E,4,0,36,85,B12,'Regular',4792,R11,1456.55,"(35, 45]","(80, 100]",0.139578,1893.518779,292.392007,336.962537
205,424.0,2,0.62,F,10,0,51,100,B12,'Regular',27000,R11,10834.00,"(45, 55]","(80, 100]",0.158238,1978.799810,341.219617,398.162354
223,463.0,1,0.31,A,5,0,45,50,B12,'Regular',12,R73,3986.67,"(35, 45]","(49, 60]",0.058121,1728.961235,128.587321,124.763448


In [3]:
df["decile"] = pd.qcut(df["prime_pure"], 10, labels=False, duplicates="drop")

resume = df.groupby("decile").apply(lambda g: pd.Series({
    "prime_pure_moy": np.average(g["prime_pure"], weights=g["Exposure"]),
    "cout_reel_moy": g["ClaimAmount"].sum() / g["Exposure"].sum(),
    "part_du_cout_total": g["ClaimAmount"].sum() / df["ClaimAmount"].sum(),
}))

resume

,prime_pure_moy,cout_reel_moy,part_du_cout_total
decile,,,
0,118.587334,96.648716,0.128690
1,128.587321,99.693128,0.116959
2,139.361030,122.265835,0.241706
3,185.540972,176.020778,0.083986
4,265.510019,282.076167,0.128483
5,418.353745,608.941658,0.300176


In [4]:
print(df["prime_pure"].nunique())

36


In [5]:
cells = (df.groupby(["age_group", "bm_group"])
           .apply(lambda g: pd.Series({
               "prime_pure": g["prime_pure"].iloc[0],
               "exposition": g["Exposure"].sum(),
               "cout": g["ClaimAmount"].sum(),
           }))
           .reset_index()
           .sort_values("prime_pure"))

cells["cout_reel"] = cells["cout"] / cells["exposition"]
cells["expo_cum"] = cells["exposition"].cumsum() / cells["exposition"].sum()
cells["groupe"] = np.ceil(cells["expo_cum"] * 10).clip(upper=10).astype(int)

show(cells)

In [6]:
resume = cells.groupby("groupe").apply(lambda c: pd.Series({
    "nb_cellules": len(c),
    "part_exposition": c["exposition"].sum() / cells["exposition"].sum(),
    "prime_pure_moy": np.average(c["prime_pure"], weights=c["exposition"]),
    "cout_reel_moy": c["cout"].sum() / c["exposition"].sum(),
    "part_du_cout_total": c["cout"].sum() / cells["cout"].sum(),
}))
resume

,nb_cellules,part_exposition,prime_pure_moy,cout_reel_moy,part_du_cout_total
groupe,,,,,
1,1.0,0.094998,113.244495,81.979798,0.046005
3,1.0,0.130409,122.479400,107.334493,0.082685
5,1.0,0.198604,128.587321,99.693128,0.116959
6,1.0,0.113298,132.284774,123.799095,0.082855
8,2.0,0.235359,145.455379,123.606316,0.171851
9,3.0,0.102013,205.945176,215.175203,0.129666
10,27.0,0.125319,373.274611,499.783356,0.369980


In [7]:
cells["part_expo"] = cells["exposition"] / cells["exposition"].sum()
cells.sort_values("part_expo", ascending=False).head(5)[["age_group", "bm_group", "part_expo"]]

,age_group,bm_group,part_expo
21,"(45, 55]","(49, 60]",0.221361
15,"(35, 45]","(49, 60]",0.198604
27,"(55, 65]","(49, 60]",0.130409
33,"(65, 100]","(49, 60]",0.113298
9,"(25, 35]","(49, 60]",0.094998


In [8]:
cells = (df.groupby(["age_group", "bm_group"])
           .apply(lambda g: pd.Series({
               "nb_polices": len(g),
               "exposition": g["Exposure"].sum(),
               "nb_sinistres": g["ClaimNb"].sum(),
               "cout": g["ClaimAmount"].sum(),
               "prime_pure": g["prime_pure"].iloc[0],
           }))
           .reset_index())

cells["part_expo"] = cells["exposition"] / cells["exposition"].sum()
cells["cout_reel"] = cells["cout"] / cells["exposition"]
cells["ecart_reel_modele"] = cells["cout_reel"] / cells["prime_pure"] - 1

show(cells)

On vérifie la corrélation entre les deux colonnes pour voir si notre modèle classe bien les risques.

In [9]:
cells[["prime_pure", "cout_reel"]].corr(method="spearman")

,prime_pure,cout_reel
prime_pure,1.000000,0.899614
cout_reel,0.899614,1.000000


In [10]:
cells["credible"] = cells["nb_sinistres"] >= 30

print(cells["credible"].value_counts())
print(cells.groupby("credible")["part_expo"].sum())

# Les 18-25 ans, toutes tranches de bonus-malus
cells[cells["age_group"] == "(17, 25]"].sort_values("prime_pure")[
    ["bm_group", "nb_sinistres", "prime_pure", "cout_reel", "ecart_reel_modele"]]

credible
True     28
False     8
Name: count, dtype: int64
credible
False    0.000485
True     0.999515
Name: part_expo, dtype: float64


,bm_group,nb_sinistres,prime_pure,cout_reel,ecart_reel_modele
4,"(60, 80]",355.0,184.555656,157.214457,-0.148146
3,"(49, 60]",69.0,201.276701,130.935779,-0.349474
5,"(80, 100]",1499.0,414.165144,1151.859452,1.781160
0,"(100, 120]",267.0,729.551282,504.632122,-0.308298
1,"(120, 150]",193.0,1137.425264,1287.796546,0.132203
2,"(150, 250]",21.0,1493.929792,2476.402280,0.657643


In [11]:
df["sin_attendus"] = df["freq_pred"] * df["Exposure"]
att = df.groupby(["age_group", "bm_group"])["sin_attendus"].sum().reset_index()

cells = cells.merge(att, on=["age_group", "bm_group"])
cells["ratio_sin"] = cells["nb_sinistres"] / cells["sin_attendus"]
cells["z"] = (cells["nb_sinistres"] - cells["sin_attendus"]) / np.sqrt(cells["sin_attendus"])

cells[cells["credible"]].sort_values("z")[
    ["age_group", "bm_group", "nb_sinistres", "sin_attendus", "ratio_sin", "z"]]

,age_group,bm_group,nb_sinistres,sin_attendus,ratio_sin,z
16,"(35, 45]","(60, 80]",1434.0,1434.0,1.0,-1.014134e-11
27,"(55, 65]","(49, 60]",2480.0,2480.0,1.0,-9.798149e-12
21,"(45, 55]","(49, 60]",4981.0,4981.0,1.0,-7.177895e-12
33,"(65, 100]","(49, 60]",1997.0,1997.0,1.0,-6.410940e-12
0,"(17, 25]","(100, 120]",267.0,267.0,1.0,-6.167843e-12
5,"(17, 25]","(80, 100]",1499.0,1499.0,1.0,-5.602576e-12
15,"(35, 45]","(49, 60]",4085.0,4085.0,1.0,-5.563920e-12
18,"(45, 55]","(100, 120]",73.0,73.0,1.0,-3.803864e-12
12,"(35, 45]","(100, 120]",127.0,127.0,1.0,-3.412292e-12
6,"(25, 35]","(100, 120]",239.0,239.0,1.0,-3.386420e-12


In [12]:
cells.groupby("age_group", observed=True)[["nb_sinistres", "sin_attendus"]].sum()

,nb_sinistres,sin_attendus
age_group,,
"(17, 25]",2404.0,2404.0
"(25, 35]",5167.0,5167.0
"(35, 45]",6315.0,6315.0
"(45, 55]",6622.0,6622.0
"(55, 65]",3316.0,3316.0
"(65, 100]",2621.0,2621.0


j'ai modifié le calcul de la prime pure finalement !

In [13]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/portfolio_priced.parquet")

cells = (df.groupby(["age_group", "bm_group"])
           .agg(exposition=("Exposure", "sum"),
                nb_sinistres=("ClaimNb", "sum"),
                cout=("ClaimAmount", "sum"),
                prime_pure=("prime_pure", "first"))
           .reset_index())

cells["part_expo"] = cells["exposition"] / cells["exposition"].sum()
cells["cout_reel"] = cells["cout"] / cells["exposition"]
cells["credible"] = cells["nb_sinistres"] >= 30

prime_moy = np.average(cells["prime_pure"], weights=cells["exposition"])
cells["multiple"] = cells["prime_pure"] / prime_moy

cells.sort_values("prime_pure", ascending=False)[
    ["age_group", "bm_group", "nb_sinistres", "part_expo", "prime_pure", "multiple", "credible"]]

,age_group,bm_group,nb_sinistres,part_expo,prime_pure,multiple,credible
26,"(55, 65]","(150, 250]",6,0.000015,2505.533786,14.801326,False
32,"(65, 100]","(150, 250]",9,0.000028,2316.631136,13.685392,False
2,"(17, 25]","(150, 250]",21,0.000104,1493.929792,8.825322,False
1,"(17, 25]","(120, 150]",193,0.001114,1137.425264,6.719288,True
20,"(45, 55]","(150, 250]",5,0.000030,1061.927410,6.273287,False
25,"(55, 65]","(120, 150]",25,0.000142,957.191351,5.654564,False
30,"(65, 100]","(100, 120]",39,0.000254,954.500597,5.638668,True
8,"(25, 35]","(150, 250]",7,0.000049,931.950647,5.505456,False
31,"(65, 100]","(120, 150]",10,0.000074,884.206906,5.223412,False
14,"(35, 45]","(150, 250]",6,0.000043,846.423879,5.000210,False


In [14]:
# Vérifie d'abord que l'ancien prix est bien dans le fichier et qu'il diffère du nouveau
print("prime_pure_v2" in df.columns, (df["prime_pure_v2"] != df["prime_pure"]).any())

ancien = df.groupby(["age_group", "bm_group"])["prime_pure_v2"].first().reset_index()
cells = cells.merge(ancien, on=["age_group", "bm_group"])

cells["prime_finale"] = np.where(cells["credible"], cells["prime_pure"], cells["prime_pure_v2"])
prime_moy = np.average(cells["prime_finale"], weights=cells["exposition"])
cells["multiple"] = cells["prime_finale"] / prime_moy

cells.sort_values("multiple", ascending=False)[
    ["age_group", "bm_group", "nb_sinistres", "part_expo", "prime_finale", "multiple", "credible"]
].head(12)

True True


,age_group,bm_group,nb_sinistres,part_expo,prime_finale,multiple,credible
20,"(45, 55]","(150, 250]",5,0.000030,1557.673458,9.201159,False
32,"(65, 100]","(150, 250]",9,0.000028,1531.050240,9.043896,False
2,"(17, 25]","(150, 250]",21,0.000104,1450.642287,8.568927,False
26,"(55, 65]","(150, 250]",6,0.000015,1335.771246,7.890385,False
14,"(35, 45]","(150, 250]",6,0.000043,1304.717740,7.706952,False
1,"(17, 25]","(120, 150]",193,0.001114,1137.425264,6.718758,True
31,"(65, 100]","(120, 150]",10,0.000074,1104.882259,6.526526,False
8,"(25, 35]","(150, 250]",7,0.000049,1033.274117,6.103538,False
25,"(55, 65]","(120, 150]",25,0.000142,964.975417,5.700098,False
30,"(65, 100]","(100, 120]",39,0.000254,954.500597,5.638224,True


In [15]:
def decision(m):
    if m <= 3:
        return "accepter"
    elif m <= 5:
        return "surprime"
    return "refuser"

cells["decision"] = cells["multiple"].apply(decision)

tot_expo = cells["exposition"].sum()
tot_cout = cells["cout"].sum()

impact = cells.groupby("decision").agg(
    nb_cellules=("decision", "size"),
    exposition=("exposition", "sum"),
    cout=("cout", "sum"),
)
impact["part_expo"] = impact["exposition"] / tot_expo
impact["part_cout"] = impact["cout"] / tot_cout
impact

,nb_cellules,exposition,cout,part_expo,part_cout
decision,,,,,
accepter,18,350346.064493,57349286.94,0.989975,0.957270
refuser,10,655.433046,795079.92,0.001852,0.013271
surprime,8,2892.490493,1764849.64,0.008173,0.029459
